In [ ]:
!pip install -q json5 json-repair


# AI SecOps Multi-Agent (LangGraph)

WAF/EDR 로그 → **Parser** → **Classifier** → **Red ∥ Blue** → **Judge** → **MITRE** → **Report**

> [3강] 랭그래프 응용 패턴: **Prompt Chaining** + **Parallelization** + **Evaluator-Optimizer** 조합


In [ ]:
# 설치
!pip install -q langgraph langchain langchain-openai langchain-google-genai pydantic grandalf


In [ ]:

# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# 환경 설정
import os

LLM_PROVIDER = "mock"  # @param ["mock", "openai", "gemini"]
GEMINI_MODEL = "gemini-2.0-flash"
OPENAI_MODEL = "gpt-4o-mini"

os.environ["LLM_PROVIDER"] = LLM_PROVIDER

def _load_secret(name: str) -> str:
    """Colab Secrets → userdata, 로컬 → getpass."""
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        from getpass import getpass
        return getpass(f"{name}: ")

if LLM_PROVIDER == "openai":
    os.environ["OPENAI_API_KEY"] = _load_secret("OPENAI_API_KEY")
elif LLM_PROVIDER == "gemini":
    os.environ["GOOGLE_API_KEY"] = _load_secret("GOOGLE_API_KEY")

print("LLM_PROVIDER:", LLM_PROVIDER)


## 강의 틀 vs AI SecOps — 뭐가 다른가?

| 항목 | [3강] 기본 틀 | AI SecOps 추가/변경 |
|------|-------------|---------------------|
| **Model 정의** | `init_chat_model()` | 동일 + **Pydantic** 구조화 출력 (판정·IOC·리포트) |
| **State** | `topic`, `draft_joke` 등 문자열 | `parsed_log`, `verdict` 등 **객체** 필드 |
| **Node** | LLM 호출만 | **parse_node** = LLM 없음 (정규식·IOC) ← **추가** |
| **그래프** | 일직선 Chain | classify 이후 **Red/Blue 병렬** → judge 합류 |
| **패턴** | Prompt Chaining | Chaining + **Parallelization** + **Evaluator**(Judge) |
| **입력** | `{"topic": "..."}` | WAF/EDR **로그 JSON** |

→ **틀 자체는 동일** (Model → State → Node → Graph → 실행). 노드 내용과 엣지(병렬)만 보안 도메인에 맞게 확장.


## Model 정의

Agent 간 주고받는 **데이터 형식** (Pydantic). 강의의 State 필드 타입을 구조화한 것.


In [ ]:
from enum import Enum
from typing import Any
from pydantic import BaseModel, Field


class LogType(str, Enum):
    WAF = "waf"
    EDR = "edr"
    AUTO = "auto"


class IOC(BaseModel):
    type: str
    value: str


class ParsedLog(BaseModel):
    log_type: str
    raw: str
    normalized: dict[str, Any] = Field(default_factory=dict)
    iocs: list[IOC] = Field(default_factory=list)


class ClassificationResult(BaseModel):
    is_attack: bool
    confidence: float
    attack_type: str = ""
    summary: str = ""


class RedTeamAnalysis(BaseModel):
    attack_hypothesis: str
    evidence: list[str] = Field(default_factory=list)
    confidence: float = 0.5


class BlueTeamAnalysis(BaseModel):
    benign_hypothesis: str
    evidence: list[str] = Field(default_factory=list)
    confidence: float = 0.5


class JudgeVerdict(BaseModel):
    verdict: str          # attack | normal | investigate
    tp_fp: str            # TP | FP | 추가조사
    confidence: float
    rationale: str        # 한 줄 요약
    final_rationale: str = ""  # LLM 최종 판단 근거 (상세)


class MitreMapping(BaseModel):
    technique_ids: list[str] = Field(default_factory=list)
    recommendations: list[str] = Field(default_factory=list)
    severity: str = "medium"


class IncidentReport(BaseModel):
    title: str
    executive_summary: str
    full_report_markdown: str = ""


## Parser (Node에서 사용)

`parse_node` 전용. LLM 호출 없이 로그 정규화 + IOC 추출.


In [ ]:
import json
import re

IP_RE = re.compile(r"\b(?:\d{1,3}\.){3}\d{1,3}\b")
URL_RE = re.compile(r"https?://\S+|^/\S+", re.I)


def parse_log(raw_log: str | dict, log_type: str = "auto") -> ParsedLog:
    raw = json.dumps(raw_log, ensure_ascii=False) if isinstance(raw_log, dict) else raw_log.strip()
    data = raw_log if isinstance(raw_log, dict) else {"message": raw}

    iocs = []
    for ip in IP_RE.findall(raw):
        iocs.append(IOC(type="ip", value=ip))
    uri = data.get("uri", data.get("url", ""))
    if uri:
        iocs.append(IOC(type="url", value=uri))

    lt = log_type
    if lt == "auto":
        lt = "edr" if any(k in raw.lower() for k in ("process", "command_line", "powershell")) else "waf"

    return ParsedLog(log_type=lt, raw=raw, normalized=data, iocs=iocs)


## Model 정의 (LLM)

강의와 동일하게 `init_chat_model`. mock이면 가짜 LLM 사용.


In [ ]:
import json

def log_is_attack(raw: str) -> bool:
    t = raw.lower()
    return any(k in t for k in ATTACK_KW)


def _to_text(content: Any) -> str:
    if content is None:
        return ""
    if isinstance(content, str):
        return content
    if isinstance(content, dict):
        if content.get("type") == "text" and "text" in content:
            return str(content["text"])
        return json.dumps(content, ensure_ascii=False)
    if isinstance(content, list):
        parts: list[str] = []
        for block in content:
            if isinstance(block, str):
                parts.append(block)
            elif isinstance(block, dict):
                parts.append(str(block.get("text") or block.get("content") or ""))
            elif hasattr(block, "text"):
                parts.append(str(block.text or ""))
            else:
                parts.append(str(block))
        return "".join(parts)
    return str(content)


def _strip_code_fence(text: str) -> str:
    text = text.strip()
    if not text.startswith("```"):
        return text
    lines = text.splitlines()
    body = lines[1:-1] if lines and lines[-1].startswith("```") else lines[1:]
    if body and body[0].strip().lower() == "json":
        body = body[1:]
    return "\n".join(body).strip()


def _parse_json_text(text: str) -> dict:
    """LLM 응답 → dict. json → json5 → json_repair 순으로 시도."""
    last_err = None
    raw = _strip_code_fence(text)
    candidates = [raw]
    start, end = raw.find("{"), raw.rfind("}")
    if start >= 0 and end > start:
        candidates.append(raw[start : end + 1])

    loaders = [("json", json.loads)]
    try:
        import json5
        loaders.append(("json5", json5.loads))
    except ImportError:
        pass
    try:
        from json_repair import repair_json
        loaders.append(("json_repair", lambda s: json.loads(repair_json(s))))
    except ImportError:
        pass

    for cand in candidates:
        for _, loader in loaders:
            try:
                data = loader(cand)
                if isinstance(data, dict):
                    return data
            except Exception as e:
                last_err = e
    if isinstance(last_err, json.JSONDecodeError):
        raise last_err
    raise json.JSONDecodeError(str(last_err or "JSON parse failed"), raw, 0)


def get_chat_model():
    provider = os.environ.get("LLM_PROVIDER", "mock")
    if provider == "mock":
        return None
    from langchain.chat_models import init_chat_model

    if provider == "gemini":
        model_name = GEMINI_MODEL
        return init_chat_model(model_name, model_provider="google_genai", temperature=0.3)
    model_name = os.environ.get("OPENAI_MODEL", "gpt-4o-mini")
    return init_chat_model(model_name, model_provider="openai", temperature=0.1)


def ask_json(model, system: str, user: str, log_raw: str = "", schema: type[BaseModel] | None = None) -> dict:
    if model is None:
        return _mock_json(system, user, log_raw)

    prompt = f"{system}\n\n{user}"

    if schema is not None:
        try:
            structured = model.with_structured_output(schema)
            out = structured.invoke(prompt)
            if isinstance(out, BaseModel):
                return out.model_dump()
            if isinstance(out, dict):
                return out
        except Exception:
            pass

    strict = (
        "\n\n반드시 유효한 JSON 객체만 출력하세요. "
        "문자열 값 안의 따옴표는 \\\" 로 이스케이프. 마크다운 코드블록 없이."
    )
    response = model.invoke(prompt + strict)
    text = _to_text(response.content)
    try:
        data = _parse_json_text(text)
    except (json.JSONDecodeError, ValueError):
        response = model.invoke(
            prompt + strict + "\n(이전 응답이 유효한 JSON이 아니었습니다. 다시 출력하세요.)"
        )
        text = _to_text(response.content)
        data = _parse_json_text(text)
    if not isinstance(data, dict):
        raise ValueError(f"LLM JSON must be dict, got {type(data)}")
    return data


def _mock_json(system: str, user: str, log_raw: str) -> dict:
    s = system.lower()
    src = (log_raw or user).lower()
    is_attack = log_is_attack(src)

    if "classification" in s:
        return {
            "is_attack": is_attack,
            "confidence": 0.85 if is_attack else 0.2,
            "attack_type": "SQL Injection" if is_attack else "",
            "summary": "공격 의심" if is_attack else "정상/저위험",
        }
    if "judge" in s:
        if is_attack:
            return {
                "verdict": "attack",
                "tp_fp": "TP",
                "confidence": 0.78,
                "rationale": "Red Team 근거 우세",
                "final_rationale": (
                    "Red Team은 로그에서 공격 패턴(의심 페이로드·비정상 URI)을 근거로 공격 가능성을 높게 평가했습니다. "
                    "Blue Team은 오탐·정상 트래픽 가능성을 제시했으나, IOC 및 1차 분류 결과와 비교할 때 공격 근거가 더 설득력 있습니다. "
                    "따라서 본 이벤트는 실제 공격(TP)으로 판단합니다."
                ),
            }
        return {
            "verdict": "normal",
            "tp_fp": "FP",
            "confidence": 0.72,
            "rationale": "Blue Team 근거 우세",
            "final_rationale": (
                "Blue Team은 업무 트래픽·오탐 가능성 및 정상 HTTP 패턴을 근거로 정상 가능성을 높게 평가했습니다. "
                "Red Team의 공격 가설은 있으나 로그만으로는 악성 행위가 명확하지 않습니다. "
                "따라서 본 이벤트는 정상/오탐(FP)으로 판단합니다."
            ),
        }
    if "red team" in s:
        return {
            "attack_hypothesis": "공격 시도 가능" if is_attack else "공격 근거 약함",
            "evidence": ["의심 페이로드", "비정상 패턴"] if is_attack else ["명확한 공격 패턴 없음"],
            "confidence": 0.8 if is_attack else 0.3,
        }
    if "blue team" in s:
        return {
            "benign_hypothesis": "정상/오탐 가능" if not is_attack else "오탐 가능성 낮음",
            "evidence": ["업무 트래픽", "차단됨"] if not is_attack else ["공격 패턴 뚜렷"],
            "confidence": 0.7 if not is_attack else 0.25,
        }
    if "mitre" in s:
        return {
            "technique_ids": ["T1190"] if is_attack else [],
            "recommendations": ["IP 차단", "WAF 룰 점검"] if is_attack else ["모니터링 유지"],
            "severity": "high" if is_attack else "low",
        }
    if "incident report" in s:
        return {
            "title": "보안 이벤트 분석 리포트",
            "executive_summary": "공격으로 분류됨" if is_attack else "정상/저위험으로 분류됨",
            "full_report_markdown": f"# Report\n\n{'Attack' if is_attack else 'Normal'}",
        }
    return {}



model = get_chat_model()
print("model:", "MockLLM" if model is None else type(model).__name__)


## State 정의

그래프 전체에서 공유하는 **공책**. 각 Node는 필요한 필드를 읽고, 결과 dict를 반환해 State에 **병합**.


In [ ]:
from typing import TypedDict, Any


class SOCState(TypedDict, total=False):
    # ── 입력 ──
    raw_log: str | dict
    log_type: str

    # ── Node 결과 (순서대로 채워짐) ──
    parsed_log: ParsedLog
    classification: ClassificationResult
    red_team: RedTeamAnalysis
    blue_team: BlueTeamAnalysis
    verdict: JudgeVerdict
    mitre: MitreMapping
    incident_report: IncidentReport


## Node 정의

각 함수: `state` 읽기 → 작업 → `{필드: 값}` 반환 (강의와 동일 패턴)


In [ ]:
# [Node 1] Parser — LLM 없음
def parse_node(state: SOCState):
    print("\n--- [1] Parser: 로그 파싱 + IOC 추출 ---")
    parsed = parse_log(state["raw_log"], state.get("log_type", "auto"))
    print(f"  log_type={parsed.log_type}, IOC {len(parsed.iocs)}개")
    return {"parsed_log": parsed}


In [ ]:
# [Node 2] Classifier
def classify_node(state: SOCState):
    print("\n--- [2] Classifier ---")
    p = state["parsed_log"]
    result = ask_json(model,
        "Attack Classification Agent. JSON: is_attack, confidence, attack_type, summary",
        f"로그:\n{p.raw[:2500]}", log_raw=p.raw, schema=ClassificationResult)
    return {"classification": ClassificationResult.model_validate(result)}


In [ ]:
# [Node 3a] Red Team
def red_team_node(state: SOCState):
    print("\n--- [3a] Red Team ---")
    p, c = state["parsed_log"], state["classification"]
    result = ask_json(model,
        "Red Team Agent. JSON: attack_hypothesis, evidence, confidence",
        f"classify={c.model_dump_json()}\n로그:\n{p.raw[:2500]}", log_raw=p.raw, schema=RedTeamAnalysis)
    return {"red_team": RedTeamAnalysis.model_validate(result)}


In [ ]:
# [Node 3b] Blue Team
def blue_team_node(state: SOCState):
    print("\n--- [3b] Blue Team ---")
    p, c = state["parsed_log"], state["classification"]
    result = ask_json(model,
        "Blue Team Agent. JSON: benign_hypothesis, evidence, confidence",
        f"classify={c.model_dump_json()}\n로그:\n{p.raw[:2500]}", log_raw=p.raw, schema=BlueTeamAnalysis)
    return {"blue_team": BlueTeamAnalysis.model_validate(result)}


In [ ]:
# [Node 4] Judge — Red/Blue 종합 + 최종 판단 근거
def judge_node(state: SOCState):
    print("\n--- [4] Judge: 최종 판정 + 판단 근거 ---")
    p = state["parsed_log"]
    c, r, b = state["classification"], state["red_team"], state["blue_team"]
    result = ask_json(model,
        """Judge Agent (SOC Lead). Red vs Blue 종합 판정.
JSON: verdict, tp_fp, confidence, rationale(한줄), final_rationale(4~6문장 한국어, Red/Blue 비교·IOC 연결)""",
        f"classify={c.model_dump_json()}\nred={r.model_dump_json()}\nblue={b.model_dump_json()}\niocs={[i.model_dump() for i in p.iocs]}\nlog={p.raw[:2000]}",
        log_raw=p.raw, schema=JudgeVerdict)
    if not result.get("final_rationale"):
        result["final_rationale"] = result.get("rationale", "")
    return {"verdict": JudgeVerdict.model_validate(result)}


In [ ]:
# [Node 5] MITRE
def mitre_node(state: SOCState):
    print("\n--- [5] MITRE ---")
    p = state["parsed_log"]
    result = ask_json(model,
        "MITRE Agent. JSON: technique_ids, recommendations, severity",
        f"verdict={state['verdict'].model_dump_json()}", log_raw=p.raw, schema=MitreMapping)
    return {"mitre": MitreMapping.model_validate(result)}


In [ ]:
# [Node 6] Report
def report_node(state: SOCState):
    print("\n--- [6] Report ---")
    p = state["parsed_log"]
    result = ask_json(model,
        "Incident Report Agent. JSON: title, executive_summary, full_report_markdown",
        f"verdict={state['verdict'].model_dump_json()}\nmitre={state['mitre'].model_dump_json()}",
        log_raw=p.raw, schema=IncidentReport)
    return {"incident_report": IncidentReport.model_validate(result)}


## 그래프 생성

```
START → parse → classify → [red_team ∥ blue_team] → judge → mitre → report → END
```


In [ ]:
from langgraph.graph import StateGraph, START, END

workflow = StateGraph(SOCState)

# 노드 등록
workflow.add_node("parse", parse_node)
workflow.add_node("classify", classify_node)
workflow.add_node("red_team", red_team_node)
workflow.add_node("blue_team", blue_team_node)
workflow.add_node("judge", judge_node)
workflow.add_node("mitre", mitre_node)
workflow.add_node("report", report_node)

# 엣지 연결
workflow.add_edge(START, "parse")
workflow.add_edge("parse", "classify")
workflow.add_edge("classify", "red_team")   # 병렬 분기
workflow.add_edge("classify", "blue_team")   # 병렬 분기
workflow.add_edge("red_team", "judge")       # 합류
workflow.add_edge("blue_team", "judge")      # 합류
workflow.add_edge("judge", "mitre")
workflow.add_edge("mitre", "report")
workflow.add_edge("report", END)

app = workflow.compile()
app


In [ ]:
# 그래프 시각화 (선택)
# from IPython.display import Markdown, display

# mermaid = app.get_graph().draw_mermaid()
# display(Markdown(f"```mermaid\n{mermaid}\n```"))


## 실행

**메인 시작:** `app.invoke(inputs)` — 강의의 `chain.invoke({"topic": ...})` 와 동일


In [ ]:
import json
import random

def dataset_row_to_log(row: dict) -> dict:
    text = row["text"]
    lines = text.strip().split("\n")
    method, uri = "GET", "/"
    if lines and " " in lines[0]:
        parts = lines[0].split(" ")
        method, uri = parts[0], (parts[1] if len(parts) > 1 else "/")
    headers = {}
    for line in lines[1:]:
        if ": " in line:
            k, v = line.split(": ", 1)
            headers[k.lower()] = v
    return {
        "source": "ai-waf-dataset",
        "http_method": method,
        "uri": uri,
        "user_agent": headers.get("user-agent", ""),
        "host": headers.get("host", ""),
        "client_ip": "198.51.100.1",
        "action": "BLOCK" if row.get("label") == "malicious" else "ALLOW",
        "message": text[:800],
    }

# 단건 테스트
WAF_LOG = {
    "source": "aws-waf", "action": "BLOCK", "http_method": "GET",
    "uri": "/api/users?id=1\' UNION SELECT username,password FROM admin--",
    "client_ip": "203.0.113.45", "message": "SQL injection blocked",
}
result = app.invoke({"raw_log": WAF_LOG, "log_type": "waf"})

# 데이터셋 배치 (100 → 10)
# data = json.load(open("/content/drive/Othercomputers/My MacBook Pro/AI-secops/test_data/waf_sample_10.json"))

# sample10 = sample100[:10]
# for i, row in enumerate(data):
#     r = app.invoke({"raw_log": dataset_row_to_log(row), "log_type": "waf"})
#     print(i+1, row["label"], r["verdict"].verdict, r["verdict"].tp_fp)


In [ ]:
# 결과 확인
v = result["verdict"]
print(f"\n{'='*60}")
print(f"【최종 판정】 {v.verdict.upper()} ({v.tp_fp})  |  confidence={v.confidence:.0%}")
print(f"{'='*60}")
print(f"\n【최종 판단 근거】 (Judge LLM 분석)")
print(v.final_rationale or v.rationale)
print(f"\n(요약) {v.rationale}")
print(f"\n--- Red Team ---\n  {result['red_team'].attack_hypothesis}")
for e in result['red_team'].evidence:
    print(f"  • {e}")
print(f"\n--- Blue Team ---\n  {result['blue_team'].benign_hypothesis}")
for e in result['blue_team'].evidence:
    print(f"  • {e}")
print(f"\nMITRE: {result['mitre'].technique_ids}")
print(f"리포트: {result['incident_report'].executive_summary}")


In [ ]:
# EDR 샘플
EDR_LOG = {
    "source": "crowdstrike",
    "process": "powershell.exe",
    "command_line": "powershell.exe -enc SQBFAFgAIAAoAE4AZQB3AC0ATwBiAGoAZQBjAHQAIABOAGUAdAAuAFcAZQBiAEMAbA",
    "hostname": "WS-042",
    "message": "Encoded PowerShell detected",
}

result_edr = app.invoke({"raw_log": EDR_LOG, "log_type": "edr"})
print(result_edr["verdict"].verdict, result_edr["verdict"].tp_fp)
